In [1]:
# Import Required Libraries
import json
import numpy as np
import pandas as pd
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from enum import Enum

# Astropy and Gaia-related imports
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table
import warnings
warnings.filterwarnings('ignore')

# Try to import astroquery Gaia
try:
    from astroquery.gaia import Gaia
    GAIA_AVAILABLE = True
except ImportError:
    print("Note: astroquery.gaia not available. Using mock data for demonstration.")
    GAIA_AVAILABLE = False

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [ ]:
from astroquery.gaia import Gaia
import pandas as pd

# 1. Define your ADQL query
# We are querying Data Release 3 (gaiadr3.gaia_source)
adql_query = f"SELECT TOP 100
    source_id, ra, dec, parallax, phot_g_mean_mag
    FROM gaiadr3.gaia_source
    WHERE parallax IS NOT NULL
    ORDER BY phot_g_mean_mag ASC
"

print("Launching query...")

# 2. Execute the query
# Use launch_job_async() for larger queries to avoid timeouts
job = Gaia.launch_job_async(adql_query)

# 3. Retrieve the results (returns an Astropy Table)
results = job.get_results()


# 4. Convert to a Pandas DataFrame for easy analysis
df = results.to_pandas()

# Display the first 5 rows
print("\nQuery successful! Here is a preview of the data:")
print(df.head())

Launching query...
500 Error 500:
null


HTTPError: Error 500:
null

In [2]:
# Define Tool Calling Functions for Gaia Data

@dataclass
class Parameter:
    """Represents a parameter for a tool function"""
    name: str
    type: str
    description: str
    required: bool = True
    default: Optional[Any] = None
    enum: Optional[List[str]] = None

@dataclass
class ToolDefinition:
    """Represents a tool definition compatible with LLM tool calling"""
    name: str
    description: str
    parameters: List[Parameter]
    return_type: str = "dict"
    
    def to_json_schema(self) -> Dict[str, Any]:
        """Convert to OpenAI-style JSON schema"""
        properties = {}
        required = []
        
        for param in self.parameters:
            prop = {
                "type": param.type,
                "description": param.description
            }
            if param.enum:
                prop["enum"] = param.enum
            if param.default is not None:
                prop["default"] = param.default
            properties[param.name] = prop
            if param.required:
                required.append(param.name)
        
        return {
            "name": self.name,
            "description": self.description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required
            }
        }

# Tool Registry - stores all available tools
tool_registry: Dict[str, ToolDefinition] = {}

def register_tool(tool_def: ToolDefinition):
    """Register a tool in the registry"""
    tool_registry[tool_def.name] = tool_def
    print(f"✓ Registered tool: {tool_def.name}")

print("✓ Tool definition classes created")

✓ Tool definition classes created


In [3]:
# Create Star Brightness Query Function

def find_brightest_stars(
    limit: int = 10,
    magnitude_range: Tuple[float, float] = (-5.0, 10.0),
    magnitude_type: str = "phot_g_mean_mag"
) -> Dict[str, Any]:
    """
    Query the brightest stars from the Gaia dataset.
    
    Args:
        limit: Maximum number of results to return
        magnitude_range: (min_mag, max_mag) tuple for filtering
        magnitude_type: Type of magnitude ('phot_g_mean_mag', 'phot_rp_mean_mag', 'phot_bp_mean_mag')
    
    Returns:
        Dictionary with star data and metadata
    """
    try:
        if not GAIA_AVAILABLE:
            return _mock_brightest_stars(limit, magnitude_range)
        
        min_mag, max_mag = magnitude_range
        query = f"""
        SELECT TOP {limit}
            source_id,
            ra,
            dec,
            {magnitude_type},
            parallax,
            parallax_error,
            pmra,
            pmdec,
            teff_gspphot
        FROM gaia.gaia_source
        WHERE {magnitude_type} BETWEEN {min_mag} AND {max_mag}
        AND {magnitude_type} IS NOT NULL
        ORDER BY {magnitude_type} ASC
        """
        
        job = Gaia.launch_job_async(query)
        results = job.get_results()
        
        return {
            "status": "success",
            "function": "find_brightest_stars",
            "count": len(results),
            "data": results.to_pandas().to_dict('records'),
            "parameters": {
                "limit": limit,
                "magnitude_range": magnitude_range,
                "magnitude_type": magnitude_type
            }
        }
    except Exception as e:
        return {
            "status": "error",
            "function": "find_brightest_stars",
            "error": str(e)
        }

# Register the tool
brightest_stars_tool = ToolDefinition(
    name="find_brightest_stars",
    description="Query the brightest stars from the Gaia dataset. Returns stars sorted by magnitude.",
    parameters=[
        Parameter(name="limit", type="integer", description="Maximum number of stars to return (default: 10)", required=False, default=10),
        Parameter(name="magnitude_range_min", type="number", description="Minimum magnitude (brighter limit)", required=False, default=-5.0),
        Parameter(name="magnitude_range_max", type="number", description="Maximum magnitude (fainter limit)", required=False, default=10.0),
        Parameter(name="magnitude_type", type="string", description="Type of magnitude to use", 
                 required=False, default="phot_g_mean_mag",
                 enum=["phot_g_mean_mag", "phot_rp_mean_mag", "phot_bp_mean_mag"])
    ],
    return_type="dict"
)
register_tool(brightest_stars_tool)

def _mock_brightest_stars(limit: int, mag_range: Tuple) -> Dict[str, Any]:
    """Mock data for testing without Gaia connection"""
    np.random.seed(42)
    mock_data = []
    for i in range(min(limit, 5)):
        mock_data.append({
            "source_id": 1000000000000 + i,
            "ra": np.random.uniform(0, 360),
            "dec": np.random.uniform(-90, 90),
            "phot_g_mean_mag": np.random.uniform(mag_range[0], mag_range[1]),
            "parallax": np.random.uniform(1, 100),
            "parallax_error": np.random.uniform(0.01, 1),
            "pmra": np.random.uniform(-100, 100),
            "pmdec": np.random.uniform(-100, 100),
            "teff_gspphot": np.random.uniform(3000, 10000)
        })
    return {
        "status": "success (mock data)",
        "function": "find_brightest_stars",
        "count": len(mock_data),
        "data": mock_data
    }

print("✓ find_brightest_stars function created")

✓ Registered tool: find_brightest_stars
✓ find_brightest_stars function created


In [4]:
# Create Star Proximity Search Function

def query_stars_by_coordinates(
    ra: float,
    dec: float,
    radius_arcmin: float = 10.0,
    limit: int = 100
) -> Dict[str, Any]:
    """
    Query stars within a specified radius of celestial coordinates.
    
    Args:
        ra: Right Ascension in degrees
        dec: Declination in degrees
        radius_arcmin: Search radius in arcminutes
        limit: Maximum number of results
    
    Returns:
        Dictionary with star data within the specified region
    """
    try:
        if not GAIA_AVAILABLE:
            return _mock_proximity_search(ra, dec, radius_arcmin, limit)
        
        radius_deg = radius_arcmin / 60.0
        query = f"""
        SELECT TOP {limit}
            source_id,
            ra,
            dec,
            phot_g_mean_mag,
            parallax,
            parallax_error,
            pmra,
            pmdec,
            SQRT(POWER(ra - {ra}, 2) + POWER(dec - {dec}, 2)) as angular_distance
        FROM gaia.gaia_source
        WHERE 1=CONTAINS(
            POINT('ICRS', ra, dec),
            CIRCLE('ICRS', {ra}, {dec}, {radius_deg})
        )
        ORDER BY angular_distance ASC
        """
        
        job = Gaia.launch_job_async(query)
        results = job.get_results()
        
        return {
            "status": "success",
            "function": "query_stars_by_coordinates",
            "count": len(results),
            "data": results.to_pandas().to_dict('records'),
            "parameters": {
                "ra": ra,
                "dec": dec,
                "radius_arcmin": radius_arcmin,
                "limit": limit
            }
        }
    except Exception as e:
        return {
            "status": "error",
            "function": "query_stars_by_coordinates",
            "error": str(e)
        }

# Register the tool
proximity_search_tool = ToolDefinition(
    name="query_stars_by_coordinates",
    description="Find stars within a specified circular region of the sky. Useful for studying stellar populations in specific regions.",
    parameters=[
        Parameter(name="ra", type="number", description="Right Ascension in degrees (0-360)", required=True),
        Parameter(name="dec", type="number", description="Declination in degrees (-90 to 90)", required=True),
        Parameter(name="radius_arcmin", type="number", description="Search radius in arcminutes", required=False, default=10.0),
        Parameter(name="limit", type="integer", description="Maximum number of results", required=False, default=100)
    ],
    return_type="dict"
)
register_tool(proximity_search_tool)

def _mock_proximity_search(ra: float, dec: float, radius: float, limit: int) -> Dict[str, Any]:
    """Mock data for proximity search"""
    np.random.seed(42)
    mock_data = []
    for i in range(min(limit, 8)):
        mock_data.append({
            "source_id": 2000000000000 + i,
            "ra": ra + np.random.uniform(-radius/60, radius/60),
            "dec": dec + np.random.uniform(-radius/60, radius/60),
            "phot_g_mean_mag": np.random.uniform(5, 15),
            "parallax": np.random.uniform(1, 50),
            "parallax_error": np.random.uniform(0.05, 1),
            "pmra": np.random.uniform(-200, 200),
            "pmdec": np.random.uniform(-200, 200),
            "angular_distance": np.random.uniform(0, radius/60)
        })
    return {
        "status": "success (mock data)",
        "function": "query_stars_by_coordinates",
        "count": len(mock_data),
        "data": mock_data
    }

print("✓ query_stars_by_coordinates function created")

✓ Registered tool: query_stars_by_coordinates
✓ query_stars_by_coordinates function created


In [5]:
# Create Parallax-Based Distance Calculation Function

def query_stars_by_parallax(
    parallax_range: Tuple[float, float] = (1.0, 100.0),
    parallax_error_max: float = 1.0,
    limit: int = 50
) -> Dict[str, Any]:
    """
    Calculate stellar distances using parallax measurements from Gaia.
    Parallax in milliarcseconds; distance = 1000 / parallax (in parsecs).
    
    Args:
        parallax_range: (min_parallax, max_parallax) in milliarcseconds
        parallax_error_max: Maximum parallax error to accept
        limit: Maximum number of results
    
    Returns:
        Dictionary with stars and calculated distances
    """
    try:
        if not GAIA_AVAILABLE:
            return _mock_parallax_query(parallax_range, limit)
        
        min_parallax, max_parallax = parallax_range
        query = f"""
        SELECT TOP {limit}
            source_id,
            ra,
            dec,
            phot_g_mean_mag,
            parallax,
            parallax_error,
            1000.0 / parallax as distance_pc,
            1000.0 / parallax / 3.26156 as distance_ly,
            pmra,
            pmdec
        FROM gaia.gaia_source
        WHERE parallax BETWEEN {min_parallax} AND {max_parallax}
        AND parallax_error < {parallax_error_max}
        AND parallax_error IS NOT NULL
        AND parallax > 0
        ORDER BY distance_pc ASC
        """
        
        job = Gaia.launch_job_async(query)
        results = job.get_results()
        
        return {
            "status": "success",
            "function": "query_stars_by_parallax",
            "count": len(results),
            "data": results.to_pandas().to_dict('records'),
            "parameters": {
                "parallax_range": parallax_range,
                "parallax_error_max": parallax_error_max,
                "limit": limit
            }
        }
    except Exception as e:
        return {
            "status": "error",
            "function": "query_stars_by_parallax",
            "error": str(e)
        }

# Register the tool
parallax_tool = ToolDefinition(
    name="query_stars_by_parallax",
    description="Find stars within a specified parallax (distance) range. Returns distances in parsecs and light-years.",
    parameters=[
        Parameter(name="parallax_min", type="number", description="Minimum parallax in milliarcseconds", required=False, default=1.0),
        Parameter(name="parallax_max", type="number", description="Maximum parallax in milliarcseconds", required=False, default=100.0),
        Parameter(name="parallax_error_max", type="number", description="Maximum parallax error tolerance", required=False, default=1.0),
        Parameter(name="limit", type="integer", description="Maximum number of results", required=False, default=50)
    ],
    return_type="dict"
)
register_tool(parallax_tool)

def _mock_parallax_query(parallax_range: Tuple, limit: int) -> Dict[str, Any]:
    """Mock parallax query data"""
    np.random.seed(42)
    mock_data = []
    for i in range(min(limit, 6)):
        parallax = np.random.uniform(parallax_range[0], parallax_range[1])
        mock_data.append({
            "source_id": 3000000000000 + i,
            "ra": np.random.uniform(0, 360),
            "dec": np.random.uniform(-90, 90),
            "phot_g_mean_mag": np.random.uniform(5, 12),
            "parallax": parallax,
            "parallax_error": np.random.uniform(0.01, 0.5),
            "distance_pc": 1000.0 / parallax,
            "distance_ly": (1000.0 / parallax) / 3.26156,
            "pmra": np.random.uniform(-50, 50),
            "pmdec": np.random.uniform(-50, 50)
        })
    return {
        "status": "success (mock data)",
        "function": "query_stars_by_parallax",
        "count": len(mock_data),
        "data": mock_data
    }

print("✓ query_stars_by_parallax function created")

✓ Registered tool: query_stars_by_parallax
✓ query_stars_by_parallax function created


In [6]:
# Create Proper Motion Analysis Function

def query_stars_by_proper_motion(
    pm_threshold: float = 100.0,
    limit: int = 50
) -> Dict[str, Any]:
    """
    Identify stars with high proper motion (fast-moving stars).
    Proper motion is measured in milliarcseconds per year.
    
    Args:
        pm_threshold: Minimum proper motion magnitude (mas/yr)
        limit: Maximum number of results
    
    Returns:
        Dictionary with high proper motion stars sorted by speed
    """
    try:
        if not GAIA_AVAILABLE:
            return _mock_proper_motion_query(pm_threshold, limit)
        
        query = f"""
        SELECT TOP {limit}
            source_id,
            ra,
            dec,
            phot_g_mean_mag,
            parallax,
            pmra,
            pmdec,
            SQRT(POWER(pmra, 2) + POWER(pmdec, 2)) as total_proper_motion,
            ATAN2(pmdec, pmra) * 180 / PI() as pm_direction_deg
        FROM gaia.gaia_source
        WHERE SQRT(POWER(pmra, 2) + POWER(pmdec, 2)) > {pm_threshold}
        AND pmra IS NOT NULL
        AND pmdec IS NOT NULL
        ORDER BY total_proper_motion DESC
        """
        
        job = Gaia.launch_job_async(query)
        results = job.get_results()
        
        return {
            "status": "success",
            "function": "query_stars_by_proper_motion",
            "count": len(results),
            "data": results.to_pandas().to_dict('records'),
            "parameters": {
                "pm_threshold": pm_threshold,
                "limit": limit
            }
        }
    except Exception as e:
        return {
            "status": "error",
            "function": "query_stars_by_proper_motion",
            "error": str(e)
        }

# Register the tool
proper_motion_tool = ToolDefinition(
    name="query_stars_by_proper_motion",
    description="Find stars with high proper motion (fast-moving stars). Useful for detecting nearby or moving stellar objects.",
    parameters=[
        Parameter(name="pm_threshold", type="number", description="Minimum proper motion magnitude in mas/yr", required=False, default=100.0),
        Parameter(name="limit", type="integer", description="Maximum number of results", required=False, default=50)
    ],
    return_type="dict"
)
register_tool(proper_motion_tool)

def _mock_proper_motion_query(pm_threshold: float, limit: int) -> Dict[str, Any]:
    """Mock proper motion query data"""
    np.random.seed(42)
    mock_data = []
    for i in range(min(limit, 7)):
        pmra = np.random.uniform(-pm_threshold, pm_threshold) * 1.5
        pmdec = np.random.uniform(-pm_threshold, pm_threshold) * 1.5
        total_pm = np.sqrt(pmra**2 + pmdec**2)
        mock_data.append({
            "source_id": 4000000000000 + i,
            "ra": np.random.uniform(0, 360),
            "dec": np.random.uniform(-90, 90),
            "phot_g_mean_mag": np.random.uniform(6, 14),
            "parallax": np.random.uniform(5, 50),
            "pmra": pmra,
            "pmdec": pmdec,
            "total_proper_motion": total_pm,
            "pm_direction_deg": np.arctan2(pmdec, pmra) * 180 / np.pi
        })
    return {
        "status": "success (mock data)",
        "function": "query_stars_by_proper_motion",
        "count": len(mock_data),
        "data": mock_data
    }

print("✓ query_stars_by_proper_motion function created")

✓ Registered tool: query_stars_by_proper_motion
✓ query_stars_by_proper_motion function created


In [7]:
# Create Magnitude Range Filter Function

def query_stars_by_magnitude_and_color(
    magnitude_min: float = 0.0,
    magnitude_max: float = 15.0,
    color_index_min: Optional[float] = None,
    color_index_max: Optional[float] = None,
    limit: int = 100
) -> Dict[str, Any]:
    """
    Filter stars by magnitude range and optionally by color indices.
    Color index = BP magnitude - RP magnitude (useful for determining stellar temperature).
    
    Args:
        magnitude_min: Minimum apparent magnitude (G band)
        magnitude_max: Maximum apparent magnitude (G band)
        color_index_min: Minimum color index (BP - RP)
        color_index_max: Maximum color index (BP - RP)
        limit: Maximum number of results
    
    Returns:
        Dictionary with filtered stars
    """
    try:
        if not GAIA_AVAILABLE:
            return _mock_magnitude_color_query(magnitude_min, magnitude_max, limit)
        
        where_clauses = [
            f"phot_g_mean_mag BETWEEN {magnitude_min} AND {magnitude_max}",
            "phot_g_mean_mag IS NOT NULL"
        ]
        
        if color_index_min is not None and color_index_max is not None:
            where_clauses.append(f"(phot_bp_mean_mag - phot_rp_mean_mag) BETWEEN {color_index_min} AND {color_index_max}")
        
        where_clause = " AND ".join(where_clauses)
        
        query = f"""
        SELECT TOP {limit}
            source_id,
            ra,
            dec,
            phot_g_mean_mag,
            phot_bp_mean_mag,
            phot_rp_mean_mag,
            (phot_bp_mean_mag - phot_rp_mean_mag) as color_index_bp_rp,
            parallax,
            teff_gspphot,
            logg_gspphot
        FROM gaia.gaia_source
        WHERE {where_clause}
        ORDER BY phot_g_mean_mag ASC
        """
        
        job = Gaia.launch_job_async(query)
        results = job.get_results()
        
        return {
            "status": "success",
            "function": "query_stars_by_magnitude_and_color",
            "count": len(results),
            "data": results.to_pandas().to_dict('records'),
            "parameters": {
                "magnitude_min": magnitude_min,
                "magnitude_max": magnitude_max,
                "color_index_min": color_index_min,
                "color_index_max": color_index_max,
                "limit": limit
            }
        }
    except Exception as e:
        return {
            "status": "error",
            "function": "query_stars_by_magnitude_and_color",
            "error": str(e)
        }

# Register the tool
magnitude_color_tool = ToolDefinition(
    name="query_stars_by_magnitude_and_color",
    description="Filter stars by apparent magnitude and color index. Useful for stellar classification and temperature estimation.",
    parameters=[
        Parameter(name="magnitude_min", type="number", description="Minimum apparent magnitude (G band)", required=False, default=0.0),
        Parameter(name="magnitude_max", type="number", description="Maximum apparent magnitude (G band)", required=False, default=15.0),
        Parameter(name="color_index_min", type="number", description="Minimum color index (BP - RP)", required=False),
        Parameter(name="color_index_max", type="number", description="Maximum color index (BP - RP)", required=False),
        Parameter(name="limit", type="integer", description="Maximum number of results", required=False, default=100)
    ],
    return_type="dict"
)
register_tool(magnitude_color_tool)

def _mock_magnitude_color_query(mag_min: float, mag_max: float, limit: int) -> Dict[str, Any]:
    """Mock magnitude and color query data"""
    np.random.seed(42)
    mock_data = []
    for i in range(min(limit, 10)):
        mock_data.append({
            "source_id": 5000000000000 + i,
            "ra": np.random.uniform(0, 360),
            "dec": np.random.uniform(-90, 90),
            "phot_g_mean_mag": np.random.uniform(mag_min, mag_max),
            "phot_bp_mean_mag": np.random.uniform(mag_min - 1, mag_max + 1),
            "phot_rp_mean_mag": np.random.uniform(mag_min - 1, mag_max + 1),
            "color_index_bp_rp": np.random.uniform(-0.5, 3.0),
            "parallax": np.random.uniform(1, 30),
            "teff_gspphot": np.random.uniform(3000, 10000),
            "logg_gspphot": np.random.uniform(2, 5)
        })
    return {
        "status": "success (mock data)",
        "function": "query_stars_by_magnitude_and_color",
        "count": len(mock_data),
        "data": mock_data
    }

print("✓ query_stars_by_magnitude_and_color function created")

✓ Registered tool: query_stars_by_magnitude_and_color
✓ query_stars_by_magnitude_and_color function created


In [8]:
# Display Tool Definitions for LLM

def print_tool_definitions():
    """Print all registered tools in JSON schema format"""
    print("=" * 80)
    print("AVAILABLE TOOLS FOR LLM TOOL CALLING")
    print("=" * 80)
    
    tools_json = []
    for tool_name, tool_def in tool_registry.items():
        tools_json.append(tool_def.to_json_schema())
    
    print(json.dumps(tools_json, indent=2))
    print("\n" + "=" * 80)
    print(f"Total tools registered: {len(tool_registry)}")
    print("=" * 80)

# Display all tools
print_tool_definitions()

AVAILABLE TOOLS FOR LLM TOOL CALLING
[
  {
    "name": "find_brightest_stars",
    "description": "Query the brightest stars from the Gaia dataset. Returns stars sorted by magnitude.",
    "parameters": {
      "type": "object",
      "properties": {
        "limit": {
          "type": "integer",
          "description": "Maximum number of stars to return (default: 10)",
          "default": 10
        },
        "magnitude_range_min": {
          "type": "number",
          "description": "Minimum magnitude (brighter limit)",
          "default": -5.0
        },
        "magnitude_range_max": {
          "type": "number",
          "description": "Maximum magnitude (fainter limit)",
          "default": 10.0
        },
        "magnitude_type": {
          "type": "string",
          "description": "Type of magnitude to use",
          "enum": [
            "phot_g_mean_mag",
            "phot_rp_mean_mag",
            "phot_bp_mean_mag"
          ],
          "default": "phot_g_mean

# Test Tool Functions with Sample Queries

Now we'll test each tool function with sample parameters to demonstrate how they query the Gaia dataset.

In [9]:
## Test 1: Find Brightest Stars

print("\n" + "="*80)
print("TEST 1: Finding Brightest Stars")
print("="*80)

result = find_brightest_stars(limit=5, magnitude_range=(-2.0, 8.0))
print(f"\nStatus: {result['status']}")
print(f"Function: {result['function']}")
print(f"Results found: {result['count']}")

if result['count'] > 0:
    df = pd.DataFrame(result['data'])
    print("\nBrightest Stars:")
    print(df[['source_id', 'ra', 'dec', 'phot_g_mean_mag', 'parallax']].to_string())
else:
    print("No data to display")


TEST 1: Finding Brightest Stars
500 Error 500:
null

Status: error
Function: find_brightest_stars


KeyError: 'count'

In [ ]:
## Test 2: Query Stars by Coordinates (Proximity Search)

print("\n" + "="*80)
print("TEST 2: Query Stars by Coordinates")
print("="*80)
print("\nSearching for stars near Sirius (RA: 101.3 deg, Dec: -16.7 deg)")

result = query_stars_by_coordinates(ra=101.3, dec=-16.7, radius_arcmin=15.0, limit=8)
print(f"\nStatus: {result['status']}")
print(f"Function: {result['function']}")
print(f"Results found: {result['count']}")

if result['count'] > 0:
    df = pd.DataFrame(result['data'])
    print("\nStars near coordinates:")
    cols = ['source_id', 'ra', 'dec', 'phot_g_mean_mag', 'angular_distance'] if 'angular_distance' in df.columns else ['source_id', 'ra', 'dec', 'phot_g_mean_mag']
    print(df[cols].to_string())
else:
    print("No data to display")

In [ ]:
## Test 3: Query Stars by Parallax (Distance)

print("\n" + "="*80)
print("TEST 3: Query Stars by Parallax (Nearby Stars)")
print("="*80)
print("\nSearching for stars within 10 parsecs (parallax > 100 mas)")

result = query_stars_by_parallax(parallax_range=(100.0, 500.0), parallax_error_max=1.0, limit=6)
print(f"\nStatus: {result['status']}")
print(f"Function: {result['function']}")
print(f"Results found: {result['count']}")

if result['count'] > 0:
    df = pd.DataFrame(result['data'])
    print("\nNearby Stars (calculated distances):")
    cols = ['source_id', 'distance_pc', 'distance_ly', 'parallax', 'phot_g_mean_mag'] if 'distance_pc' in df.columns else ['source_id', 'parallax', 'phot_g_mean_mag']
    print(df[cols].to_string())
else:
    print("No data to display")

In [ ]:
## Test 4: Query Stars by Proper Motion

print("\n" + "="*80)
print("TEST 4: Query Stars by Proper Motion (Fast-Moving Stars)")
print("="*80)
print("\nSearching for stars with high proper motion (>150 mas/yr)")

result = query_stars_by_proper_motion(pm_threshold=150.0, limit=7)
print(f"\nStatus: {result['status']}")
print(f"Function: {result['function']}")
print(f"Results found: {result['count']}")

if result['count'] > 0:
    df = pd.DataFrame(result['data'])
    print("\nHigh Proper Motion Stars:")
    cols = ['source_id', 'total_proper_motion', 'pmra', 'pmdec', 'parallax', 'phot_g_mean_mag'] if 'total_proper_motion' in df.columns else ['source_id', 'pmra', 'pmdec', 'parallax']
    print(df[cols].to_string())
else:
    print("No data to display")

In [ ]:
## Test 5: Query Stars by Magnitude and Color

print("\n" + "="*80)
print("TEST 5: Query Stars by Magnitude and Color Index")
print("="*80)
print("\nSearching for red dwarf stars (magnitude 8-12, red color)")

result = query_stars_by_magnitude_and_color(
    magnitude_min=8.0,
    magnitude_max=12.0,
    color_index_min=1.0,
    color_index_max=3.0,
    limit=8
)
print(f"\nStatus: {result['status']}")
print(f"Function: {result['function']}")
print(f"Results found: {result['count']}")

if result['count'] > 0:
    df = pd.DataFrame(result['data'])
    print("\nRed Dwarf Stars (estimated):")
    cols = ['source_id', 'phot_g_mean_mag', 'color_index_bp_rp', 'teff_gspphot', 'parallax'] if 'color_index_bp_rp' in df.columns else ['source_id', 'phot_g_mean_mag', 'parallax']
    print(df[cols].to_string())
else:
    print("No data to display")

# Validate Tool Calling Output Format

This section verifies that all function outputs conform to the expected JSON schema required for LLM tool calling in the RAG pipeline.

In [ ]:
## Validate Output Format and Error Handling

def validate_tool_output(result: Dict[str, Any], function_name: str) -> bool:
    """
    Validate that a tool output matches expected format for LLM tool calling.
    
    Expected format:
    {
        "status": "success" | "error" | "success (mock data)",
        "function": "<function_name>",
        "count": <integer>,
        "data": <list or dict>,
        "parameters": <dict>,
        "error": <optional string if status is error>
    }
    """
    required_fields = {"status", "function", "count"}
    has_required = required_fields.issubset(set(result.keys()))
    
    if not has_required:
        return False
    
    if result.get("status") == "error":
        return "error" in result
    
    if result.get("function") != function_name:
        return False
    
    if not isinstance(result.get("count"), int):
        return False
    
    return True

# Test all functions and validate outputs
print("\n" + "="*80)
print("VALIDATION: Testing Output Formats")
print("="*80)

test_functions = [
    ("find_brightest_stars", lambda: find_brightest_stars(limit=3)),
    ("query_stars_by_coordinates", lambda: query_stars_by_coordinates(ra=100, dec=0, radius_arcmin=10, limit=3)),
    ("query_stars_by_parallax", lambda: query_stars_by_parallax(parallax_range=(50, 150), limit=3)),
    ("query_stars_by_proper_motion", lambda: query_stars_by_proper_motion(pm_threshold=100, limit=3)),
    ("query_stars_by_magnitude_and_color", lambda: query_stars_by_magnitude_and_color(magnitude_min=5, magnitude_max=10, limit=3))
]

validation_results = []
for func_name, func_call in test_functions:
    result = func_call()
    is_valid = validate_tool_output(result, func_name)
    validation_results.append({
        "function": func_name,
        "valid": is_valid,
        "status": result.get("status"),
        "count": result.get("count")
    })
    
    check_symbol = "✓" if is_valid else "✗"
    print(f"{check_symbol} {func_name:40s} | Status: {result.get('status'):20s} | Count: {result.get('count')}")

print("\n" + "="*80)
print(f"Validation Summary: {sum(1 for r in validation_results if r['valid'])}/{len(validation_results)} functions passed")
print("="*80)

# Summary and RAG Pipeline Integration

## Available Tools Overview

This notebook has successfully defined and tested 5 tool calling functions for the Gaia RAG pipeline:

1. **find_brightest_stars** - Query the brightest stars in the sky by magnitude
2. **query_stars_by_coordinates** - Find stars in a specific celestial region
3. **query_stars_by_parallax** - Identify nearby stars by distance (parallax)
4. **query_stars_by_proper_motion** - Find fast-moving stars
5. **query_stars_by_magnitude_and_color** - Filter stars by brightness and color index

## Integration with LLM Tool Calling

All tools are compatible with LLM tool calling frameworks (OpenAI, Anthropic, etc.). The tool schemas can be extracted using:

```python
# Get individual tool schema
tool_registry["find_brightest_stars"].to_json_schema()

# Get all tools as JSON
print_tool_definitions()
```

## Next Steps for RAG Pipeline

1. **Implement Function Router** - Map LLM tool calls to actual function implementations
2. **Add Error Handling** - Enhance error recovery and user-friendly messages
3. **Implement Caching** - Cache Gaia query results to reduce API calls
4. **Add Context Window** - Maintain conversation history for multi-turn queries
5. **Implement Response Formatting** - Format astronomical data for natural language responses

In [ ]:
## Bonus: Advanced Queries - Combining Multiple Tools

# Example: Find nearby, bright, fast-moving stars
def find_advanced_targets(
    brightness_limit: float = 10.0,
    distance_pc: float = 10.0,
    min_pm: float = 100.0
) -> Dict[str, Any]:
    """
    Advanced query combining multiple criteria:
    - Nearby stars (by parallax)
    - Bright stars (by magnitude)
    - Fast-moving stars (by proper motion)
    """
    # Convert distance in parsecs to parallax in milliarcseconds
    parallax_threshold = 1000.0 / distance_pc
    
    print(f"\nSearching for nearby ({distance_pc}pc), bright (<{brightness_limit}mag), fast-moving (>{min_pm}mas/yr) stars...")
    
    # Get all results
    nearby = query_stars_by_parallax(parallax_range=(parallax_threshold * 0.8, 1000.0), limit=100)
    bright = find_brightest_stars(limit=100, magnitude_range=(-5, brightness_limit))
    fast = query_stars_by_proper_motion(pm_threshold=min_pm, limit=100)
    
    results_summary = {
        "nearby_count": nearby.get("count", 0),
        "bright_count": bright.get("count", 0),
        "fast_count": fast.get("count", 0),
        "search_criteria": {
            "max_distance_pc": distance_pc,
            "max_magnitude": brightness_limit,
            "min_proper_motion_mas_yr": min_pm
        }
    }
    
    return results_summary

print("\n" + "="*80)
print("ADVANCED EXAMPLE: Multi-Criteria Star Search")
print("="*80)

advanced_results = find_advanced_targets(brightness_limit=8.0, distance_pc=20.0, min_pm=50.0)
print(json.dumps(advanced_results, indent=2))

# Reference: Tool Calling and Gaia Data

## Gaia Mission Overview
The ESA Gaia mission provides precise astrometric, photometric, and spectroscopic measurements of ~1.8 billion stars in the Milky Way.

### Key Measurements in This Notebook

| Measurement | Unit | Description |
|---|---|---|
| **Magnitude** | mag | Brightness of stars (G, BP, RP bands) |
| **Parallax** | mas | Annual shift of star position; used to calculate distance |
| **Distance** | pc, ly | Distance derived from parallax (1 pc = 3.26 light-years) |
| **Proper Motion** | mas/yr | Star's motion across the sky |
| **RA/Dec** | degrees | Celestial coordinates |
| **Color Index** | mag | BP - RP magnitude difference; indicates stellar temperature |
| **Effective Temperature** | K | Surface temperature (teff_gspphot) |
| **Surface Gravity** | log(cm/s²) | Stellar density indicator (logg_gspphot) |

## Tool Calling Framework Compatibility

These tools are designed to work with LLM tool calling frameworks including:
- OpenAI's Function Calling (GPT-4, GPT-3.5)
- Anthropic's Tool Use
- Open-source frameworks (Ollama, LM Studio)
- Custom implementations

## Example LLM Query

**User:** "Find me the 5 brightest stars in the northern sky"

**LLM Tool Call:**
```json
{
  "tool": "find_brightest_stars",
  "parameters": {
    "limit": 5,
    "magnitude_range_min": -5.0,
    "magnitude_range_max": 10.0
  }
}
```

**System Response:** Returns brightest stars with metadata

**LLM Response:** "The 5 brightest stars include Sirius (mag -1.46), Canopus (mag -0.74)..."

In [ ]:
## Implementation: LLM Tool Router and Function Executor

class ToolExecutor:
    """
    Handles tool execution for LLM-based requests.
    Maps tool calls to actual functions and manages execution.
    """
    
    def __init__(self):
        self.tools = {
            "find_brightest_stars": find_brightest_stars,
            "query_stars_by_coordinates": query_stars_by_coordinates,
            "query_stars_by_parallax": query_stars_by_parallax,
            "query_stars_by_proper_motion": query_stars_by_proper_motion,
            "query_stars_by_magnitude_and_color": query_stars_by_magnitude_and_color
        }
    
    def execute(self, tool_name: str, parameters: Dict[str, Any]) -> Dict[str, Any]:
        """Execute a tool with given parameters"""
        if tool_name not in self.tools:
            return {
                "status": "error",
                "error": f"Unknown tool: {tool_name}",
                "available_tools": list(self.tools.keys())
            }
        
        try:
            # Convert parameter names for function calls
            if tool_name == "find_brightest_stars":
                result = self.tools[tool_name](
                    limit=parameters.get("limit", 10),
                    magnitude_range=(
                        parameters.get("magnitude_range_min", -5.0),
                        parameters.get("magnitude_range_max", 10.0)
                    ),
                    magnitude_type=parameters.get("magnitude_type", "phot_g_mean_mag")
                )
            elif tool_name == "query_stars_by_coordinates":
                result = self.tools[tool_name](
                    ra=parameters.get("ra"),
                    dec=parameters.get("dec"),
                    radius_arcmin=parameters.get("radius_arcmin", 10.0),
                    limit=parameters.get("limit", 100)
                )
            elif tool_name == "query_stars_by_parallax":
                result = self.tools[tool_name](
                    parallax_range=(
                        parameters.get("parallax_min", 1.0),
                        parameters.get("parallax_max", 100.0)
                    ),
                    parallax_error_max=parameters.get("parallax_error_max", 1.0),
                    limit=parameters.get("limit", 50)
                )
            elif tool_name == "query_stars_by_proper_motion":
                result = self.tools[tool_name](
                    pm_threshold=parameters.get("pm_threshold", 100.0),
                    limit=parameters.get("limit", 50)
                )
            elif tool_name == "query_stars_by_magnitude_and_color":
                result = self.tools[tool_name](
                    magnitude_min=parameters.get("magnitude_min", 0.0),
                    magnitude_max=parameters.get("magnitude_max", 15.0),
                    color_index_min=parameters.get("color_index_min"),
                    color_index_max=parameters.get("color_index_max"),
                    limit=parameters.get("limit", 100)
                )
            
            return result
        except Exception as e:
            return {
                "status": "error",
                "function": tool_name,
                "error": str(e)
            }
    
    def get_tool_schemas(self) -> List[Dict[str, Any]]:
        """Get all tool schemas for LLM"""
        return [tool_registry[name].to_json_schema() for name in self.tools.keys()]

# Example usage
print("\n" + "="*80)
print("TOOL EXECUTOR EXAMPLE")
print("="*80)

executor = ToolExecutor()

# Simulate LLM tool call
llm_request = {
    "tool": "find_brightest_stars",
    "parameters": {
        "limit": 3,
        "magnitude_range_min": -2.0,
        "magnitude_range_max": 6.0
    }
}

print(f"\nLLM Tool Request: {llm_request['tool']}")
print(f"Parameters: {json.dumps(llm_request['parameters'], indent=2)}")

result = executor.execute(llm_request["tool"], llm_request["parameters"])
print(f"\nExecution Result:")
print(f"  Status: {result.get('status')}")
print(f"  Records: {result.get('count')}")
print(f"  Function: {result.get('function')}")

print("\n✓ Tool executor ready for LLM integration!")

# ESA Gaia Dataset Tool Calling Functions for RAG Pipeline

This notebook defines and tests tool calling functions designed for a Retrieval-Augmented Generation (RAG) pipeline using the ESA Gaia dataset. These functions enable an LLM to query the Gaia archive by executing predefined tools with specific parameters.

## Overview
- Define structured tool functions compatible with LLM tool calling
- Query brightest stars, nearest stars, and celestial objects
- Filter by magnitude, distance, proper motion, and parallax
- Test all functions with actual Gaia dataset queries